In [1]:
# ============================== Block 1 (CLEAN: paths + constants + geometry) ==============================
from pathlib import Path
import os
import re
import json

import numpy as np
import matplotlib.pyplot as plt
import flopy

# -------------------------------
# Paths (AUTHORITATIVE UNMASKED STORE)
# -------------------------------
workspace = Path(r"C:\Jupyterbook\Working\Geostat_Real\Scenario_A_nomask_30").resolve()
workspace.mkdir(parents=True, exist_ok=True)
print("workspace:", workspace)

# Where successful UNMASKED realizations are stored (this is what we COUNT)
REALZ_UNMASKED = workspace / "unmasked"
REALZ_UNMASKED.mkdir(parents=True, exist_ok=True)
print("REALZ_UNMASKED:", REALZ_UNMASKED)

# Optional: unmasked figures
figs_path = workspace / "figures_unmasked"
figs_path.mkdir(parents=True, exist_ok=True)
print("figs_path:", figs_path)

# -------------------------------
# Flags
# -------------------------------
def _get_env_bool(name: str, default: bool = True) -> bool:
    v = os.environ.get(name, str(default))
    return str(v).strip().lower() in {"1", "true", "t", "yes", "y", "on"}

WRITE = _get_env_bool("WRITE", True)
RUN   = _get_env_bool("RUN", True)
PLOT  = _get_env_bool("PLOT", True)

# -------------------------------
# TIME (MF6 uses DAYS)
# -------------------------------
TOTAL_YEARS    = 8000
INTERVAL_YEARS = 1000
nper   = TOTAL_YEARS // INTERVAL_YEARS              # 8
perlen = [365.0 * INTERVAL_YEARS] * nper            # days
nstp   = [2000] * nper                              # 2000 stabilizes ts1
tsmult = [1.0] * nper

T_FINAL = float(sum(perlen))  # expected final TOTIM in DAYS

# -------------------------------
# GRID (24 km × 300 m) cross-section, 1 m thick
# -------------------------------
nlay = 30
nrow = 1
system_length = 24_000.0
ncol = 240
delr = system_length / ncol   # 100 m
delc = 1.0                    # 1 m thickness
delv = 10.0
top  = 0.0
botm = [top - (k + 1) * delv for k in range(nlay)]  # -10 ... -300

# -------------------------------
# Solver
# -------------------------------
nouter, ninner = 800, 4000
hclose, rclose = 1e-4, 1e-2
relax          = 0.35

# -------------------------------
# Density / transport
# -------------------------------
C0       = 0.0
Cmax     = 250.0
buy_beta = 0.8

alh = 30.0
ath1 = 5.0
atv  = 0.20
diffusion_coefficient = 5.0e-9 * 86400.0   # m2/day
porosity_const = 0.30

anisotropy_v = 5.0          # Kv = Kh/anisotropy_v
Ss = 1e-4                   # <-- FIXED (was merged into anisotropy line)
Sy = 0.02

# -------------------------------
# Boundary scalars
# -------------------------------
inflow_top_thickness_m = 100.0
head_salar = -1.0
parameters = {"Scenario_A": {"inflow": 0.5}}  # m3/day per 1m thickness

# -------------------------------
# K definitions
# -------------------------------
K_catalog = {"bedrock": 1e-7}
K0_CONTINUOUS = 2.70052  # m/day (geometric mean, excl bedrock)

# -------------------------------
# Geometry helpers (used for excluding bedrock in brine IC)
# -------------------------------
def paint_bedrock_polyline(cat3d, *, top, delv, delr, nlay, ncol, x_m, z_bedrock_m, fill="below"):
    """
    Paint ONLY the bedrock category mask from a polyline into cat3d (object array).
    """
    x_m = np.asarray(x_m, float)
    z_bedrock_m = np.asarray(z_bedrock_m, float)

    x_centers = np.cumsum(np.full(ncol, delr)) - 0.5 * delr
    z_line = np.interp(x_centers, x_m, z_bedrock_m, left=z_bedrock_m[0], right=z_bedrock_m[-1])

    for j, z_b in enumerate(z_line):
        for k in range(nlay):
            zc = top - (k + 0.5) * delv
            is_bed = (zc <= z_b + 1e-9) if (fill == "below") else (zc >= z_b - 1e-9)
            if is_bed:
                cat3d[k, 0, j] = "bedrock"
    return cat3d

# -------------------------------
# Scenario A bedrock geometry
# -------------------------------
BEDROCK_START_M = 7_000.0
BEDROCK_END_M   = float(system_length)
BEDROCK_KILL_Z  = botm[-1] - 500.0

x_m_A  = [0.0, BEDROCK_START_M - 1.0, BEDROCK_START_M, 11_000.0, 14_000.0, 17_000.0, 20_000.0, BEDROCK_END_M]
zbed_A = [BEDROCK_KILL_Z, BEDROCK_KILL_Z, botm[-1], -210.0, -150.0, -115.0, -95.0, -85.0]

scenarios = {"Scenario_A": {"x": x_m_A, "z": zbed_A}}

workspace: C:\Jupyterbook\Working\Geostat_Real\Scenario_A_nomask_30
REALZ_UNMASKED: C:\Jupyterbook\Working\Geostat_Real\Scenario_A_nomask_30\unmasked
figs_path: C:\Jupyterbook\Working\Geostat_Real\Scenario_A_nomask_30\figures_unmasked


In [2]:
# ============================== Block 2 (CLEAN: unmasked K generator + cat_base) ==============================
geo_A = scenarios["Scenario_A"]

# category array: only for geometry identification (bedrock mask)
cat_base = np.full((nlay, 1, ncol), "basin", dtype=object)
cat_base = paint_bedrock_polyline(
    cat_base,
    top=top, delv=delv, delr=delr, nlay=nlay, ncol=ncol,
    x_m=geo_A["x"], z_bedrock_m=geo_A["z"],
    fill="below"
)

REAL = {
    "seed0": 73073,
    "seed_stride": 17,
    "sigma_log10": 0.5,
    "Lx_m": 1500.0,
    "Lz_m": 30.0,
}

def _gaussian_smooth_2d(arr, sig_k, sig_j):
    def _kernel(sig):
        rad = int(np.ceil(3 * sig))
        x = np.arange(-rad, rad + 1)
        w = np.exp(-(x**2) / (2 * sig**2))
        w /= w.sum()
        return w
    wk = _kernel(sig_k)
    wj = _kernel(sig_j)
    tmp = np.apply_along_axis(lambda v: np.convolve(v, wk, mode="same"), 0, arr)
    out = np.apply_along_axis(lambda v: np.convolve(v, wj, mode="same"), 1, tmp)
    return out

def _seed(i: int) -> int:
    return int(REAL["seed0"] + i * REAL["seed_stride"])

def make_K_realization_unmasked(seed: int) -> np.ndarray:
    """
    ALWAYS produces the UNMASKED (pre-bedrock-mask) K field.
    """
    rng = np.random.default_rng(int(seed))

    sig_j = max(1.0, (REAL["Lx_m"] / float(delr)) / 3.0)
    sig_k = max(1.0, (REAL["Lz_m"] / float(delv)) / 3.0)

    Z = rng.standard_normal((nlay, ncol))
    Z = _gaussian_smooth_2d(Z, sig_k=sig_k, sig_j=sig_j)
    Z = (Z - Z.mean()) / (Z.std() + 1e-12)

    log10K0 = float(np.log10(K0_CONTINUOUS))
    log10K  = log10K0 + float(REAL["sigma_log10"]) * Z
    K2 = 10 ** log10K

    K3d = np.zeros((nlay, 1, ncol), dtype=float)
    K3d[:, 0, :] = K2
    return K3d

In [3]:
# ============================== Block 2.5 (CLEAN: build_models) ==============================
def build_models(sim_folder: str, *, K3d: np.ndarray, cat3d: np.ndarray, inflow: float, geo: dict):
    sim_ws = workspace / sim_folder
    sim_ws.mkdir(parents=True, exist_ok=True)

    sim = flopy.mf6.MFSimulation(
        sim_name=sim_folder,
        exe_name="mf6.exe",
        sim_ws=sim_ws,
    )

    # --- TDIS ---
    perioddata = [(float(perlen[i]), int(nstp[i]), float(tsmult[i])) for i in range(int(nper))]
    flopy.mf6.ModflowTdis(sim, time_units="DAYS", nper=int(nper), perioddata=perioddata)

    # --- IMS: flow + trans ---
    ims_flow = flopy.mf6.ModflowIms(
        sim, pname="IMS-FLOW", filename="flow.ims",
        print_option="SUMMARY", complexity="complex",
        outer_maximum=int(nouter), outer_dvclose=float(hclose),
        inner_maximum=int(ninner), inner_dvclose=float(hclose),
        rcloserecord=float(rclose), relaxation_factor=float(relax),
    )
    ims_trans = flopy.mf6.ModflowIms(
        sim, pname="IMS-TRANS", filename="trans.ims",
        print_option="SUMMARY", complexity="moderate",
        outer_maximum=int(nouter), outer_dvclose=float(hclose),
        inner_maximum=int(ninner), inner_dvclose=float(hclose),
        rcloserecord=float(rclose), relaxation_factor=float(relax),
    )

    # --- GWF ---
    gwf = flopy.mf6.ModflowGwf(sim, modelname="flow", save_flows=True, newtonoptions="NEWTON")
    sim.register_ims_package(ims_flow, [gwf.name])

    flopy.mf6.ModflowGwfdis(
        gwf, nlay=int(nlay), nrow=int(nrow), ncol=int(ncol),
        length_units="METERS", delr=delr, delc=delc, top=top, botm=botm
    )

    k33 = (K3d / float(anisotropy_v)).tolist()
    flopy.mf6.ModflowGwfnpf(
        gwf,
        save_specific_discharge=True,
        icelltype=0,
        k=K3d.tolist(),
        k33=k33
    )

    steady = {0: False}
    transient = {kper: True for kper in range(int(nper))}
    flopy.mf6.ModflowGwfsto(
        gwf, iconvert=0, ss=float(Ss), sy=float(Sy),
        steady_state=steady, transient=transient, save_flows=True
    )

    flopy.mf6.ModflowGwfic(gwf, strt=float(head_salar))

    # --- GWT ---
    gwt = flopy.mf6.ModflowGwt(sim, modelname="trans", save_flows=True)
    sim.register_ims_package(ims_trans, [gwt.name])

    flopy.mf6.ModflowGwtdis(
        gwt, nlay=int(nlay), nrow=int(nrow), ncol=int(ncol),
        delr=delr, delc=delc, top=top, botm=botm
    )

    # ---- Initial concentration (brine pool), exclude bedrock ----
    C_ic = np.full((int(nlay), int(nrow), int(ncol)), float(C0), dtype=float)

    HALITE_X_MAX_M     = 9000.0
    HALITE_THICKNESS_M = 250.0

    j_cnc_max = min(int(HALITE_X_MAX_M / float(delr)), int(ncol) - 1)
    n_top_layers = min(int(HALITE_THICKNESS_M / float(delv)), int(nlay))

    brine_mask = np.zeros((int(nlay), 1, int(ncol)), dtype=bool)
    brine_mask[0:n_top_layers, 0, 0:j_cnc_max+1] = True
    brine_mask &= ~(cat3d == "bedrock")

    C_ic[brine_mask] = float(Cmax)
    flopy.mf6.ModflowGwtic(gwt, strt=C_ic)

    flopy.mf6.ModflowGwtmst(gwt, porosity=float(porosity_const))
    flopy.mf6.ModflowGwtadv(gwt, scheme="UPSTREAM")
    flopy.mf6.ModflowGwtdsp(
        gwt,
        alh=float(alh), ath1=float(ath1), atv=float(atv),
        diffc=float(diffusion_coefficient)
    )

    # --- FLOW BCs ---
    # (A) LEFT CHD full-depth at x=0
    chd_cells = [((k, 0, 0), float(head_salar)) for k in range(int(nlay))]
    chd_spd = {sp: chd_cells for sp in range(int(nper))}
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chd_spd, pname="CHD-LEFT", save_flows=True)

    # (B) DRN evap zone – top layer x ~ 9.5–15 km
    j_evap_start = max(0, min(int(round(9500.0 / float(delr))), int(ncol) - 1))
    j_evap_end   = max(0, min(int(round(15000.0 / float(delr))), int(ncol) - 1))
    if j_evap_end < j_evap_start:
        j_evap_start, j_evap_end = j_evap_end, j_evap_start

    drn_cells = [((0, 0, j), float(top)) for j in range(j_evap_start, j_evap_end + 1)]
    if drn_cells:
        cond_per_cell = float(inflow) / float(len(drn_cells))
        drn_spd = {
            sp: [(cid, elev, cond_per_cell, float(C0)) for (cid, elev) in drn_cells]
            for sp in range(int(nper))
        }
        flopy.mf6.ModflowGwfdrn(
            gwf, stress_period_data=drn_spd, pname="DRN-EVAP",
            auxiliary=["CONCENTRATION"], save_flows=True
        )

    # (C) RIGHT WEL inflow – upper inflow_top_thickness_m
    j_right = int(ncol) - 1
    n_in_layers = max(1, min(int(inflow_top_thickness_m / float(delv)), int(nlay)))
    inflow_cells = [(k, 0, j_right) for k in range(n_in_layers)]

    q_per_cell = float(inflow) / float(len(inflow_cells))
    wel_spd = {sp: [[cell, q_per_cell, float(C0)] for cell in inflow_cells] for sp in range(int(nper))}
    flopy.mf6.ModflowGwfwel(
        gwf, stress_period_data=wel_spd, pname="WEL-RIGHT",
        auxiliary=["CONCENTRATION"], save_flows=True
    )

    # --- TRANSPORT BCs ---
    k_idx, _, j_idx = np.where(brine_mask)
    cnc_cells = [((int(k), 0, int(j)), float(Cmax)) for k, j in zip(k_idx, j_idx)]
    cnc_spd = {sp: cnc_cells for sp in range(int(nper))}
    flopy.mf6.ModflowGwtcnc(gwt, stress_period_data=cnc_spd, pname="CNC-BRINE", save_flows=True)

    flopy.mf6.ModflowGwtssm(
        gwt,
        sources=[
            ("WEL-RIGHT", "AUX", "CONCENTRATION"),
            ("DRN-EVAP",  "AUX", "CONCENTRATION"),
        ],
        save_flows=True
    )

    # --- BUY density coupling + exchange ---
    flopy.mf6.ModflowGwfbuy(
        gwf,
        nrhospecies=1,
        denseref=1000.0,
        packagedata=[(0, float(buy_beta), 0.0, gwt.name, "CONCENTRATION")],
    )
    flopy.mf6.ModflowGwfgwt(
        sim,
        exgtype="GWF6-GWT6",
        exgmnamea=gwf.name,
        exgmnameb=gwt.name,
        filename="flow.gwfgwt"
    )

    # --- Output control ---
    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord="flow.hds",
        budget_filerecord="flow.bud",
        saverecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
        printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")]
    )
    flopy.mf6.ModflowGwtoc(
        gwt,
        concentration_filerecord="trans.ucn",
        budget_filerecord="trans.cbc",
        saverecord=[("CONCENTRATION", "LAST"), ("BUDGET", "LAST")],
        printrecord=[("CONCENTRATION", "LAST"), ("BUDGET", "LAST")]
    )

    return sim

In [ ]:
# ============================== Block 3 (CLEAN: success checks + save/plot + runner) ==============================
# -------------------------------
# MF6 success checks (TRUE success)
# -------------------------------
def lst_normal_termination(sim_ws: Path) -> bool:
    lst = sim_ws / "mfsim.lst"
    if not lst.exists():
        return False
    txt = lst.read_text(errors="ignore")
    return "normal termination" in txt.lower()

def last_time_headfile(path: Path, text: str | None = None) -> float | None:
    hf = flopy.utils.HeadFile(path, text=text) if text else flopy.utils.HeadFile(path)
    times = hf.get_times()
    return None if not times else float(times[-1])

def outputs_true_success(sim_ws: Path, t_final: float, tol: float = 1e-6) -> bool:
    hds = sim_ws / "flow.hds"
    ucn = sim_ws / "trans.ucn"
    if not (hds.exists() and ucn.exists()):
        return False
    if not lst_normal_termination(sim_ws):
        return False
    try:
        t_h = last_time_headfile(hds)
        t_c = last_time_headfile(ucn, text="CONCENTRATION")
    except Exception:
        return False
    if (t_h is None) or (t_c is None):
        return False
    return (abs(t_h - t_final) <= tol) and (abs(t_c - t_final) <= tol)

def is_partial(sim_ws: Path) -> bool:
    return (sim_ws / "mfsim.lst").exists() and (not outputs_true_success(sim_ws, T_FINAL))

# -------------------------------
# AUTHORITATIVE unmasked success counting (FROM REALZ_UNMASKED)
# -------------------------------
def list_unmasked_dirs(prefix: str) -> list[Path]:
    pat = re.compile(rf"^{re.escape(prefix)}\d{{3}}$")
    return sorted([d for d in REALZ_UNMASKED.iterdir() if d.is_dir() and pat.match(d.name)])

def count_unmasked_successes(prefix: str) -> int:
    return sum(1 for d in list_unmasked_dirs(prefix) if (d / "K3d.npy").exists())

def get_unmasked_success_names(prefix: str) -> list[str]:
    return [d.name for d in list_unmasked_dirs(prefix) if (d / "K3d.npy").exists()]

def next_new_index_from_unmasked(prefix: str) -> int:
    pat = re.compile(rf"^{re.escape(prefix)}(\d{{3}})$")
    idxs = []
    for d in REALZ_UNMASKED.iterdir():
        if d.is_dir():
            m = pat.match(d.name)
            if m:
                idxs.append(int(m.group(1)))
    return (max(idxs) + 1) if idxs else 0

# -------------------------------
# Save UNMASKED K + metadata + heatmap (ONLY AFTER TRUE SUCCESS)
# -------------------------------
def save_unmasked_realization(sim_name: str, K3d: np.ndarray, *, seed: int):
    outdir = REALZ_UNMASKED / sim_name

    # extra safety: do not overwrite existing successes
    if (outdir / "K3d.npy").exists():
        print(f"⏩ Already saved in REALZ_UNMASKED, skipping save: {sim_name}")
        return outdir

    outdir.mkdir(parents=True, exist_ok=True)
    np.save(outdir / "K3d.npy", K3d)

    meta = {
        "sim_name": sim_name,
        "seed": int(seed),
        "T_FINAL_days": float(T_FINAL),
        "REAL": dict(REAL),
        "note": "UNMASKED K field saved (pre-mask)."
    }
    (outdir / "meta.json").write_text(json.dumps(meta, indent=2))
    return outdir

def plot_K_heatmap(sim_name: str, K3d: np.ndarray, *, top_window_m: float = 300.0):
    K2 = K3d[:, 0, :]
    with np.errstate(divide="ignore", invalid="ignore"):
        logK = np.log10(K2)

    kmax = min(int(nlay), int(float(top_window_m) / float(delv)))
    logK_win = logK[:kmax, :]

    x_km = ((np.arange(ncol) + 0.5) * float(delr)) / 1000.0
    z_centers = top - (np.arange(nlay) + 0.5) * float(delv)
    z_win = z_centers[:kmax]

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(
        logK_win, aspect="auto", origin="upper",
        extent=[x_km.min(), x_km.max(), 0.0, float(z_win.min())],
    )
    ax.set_title(f"{sim_name} | log10(K) | top {int(top_window_m)} m")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("z (m)")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("log10(K)")
    fig.tight_layout()

    outpng = figs_path / f"{sim_name}_log10K_top{int(top_window_m)}m.png"
    fig.savefig(outpng, dpi=200)
    plt.close(fig)
    print("Saved:", outpng)

def write_models(sim, silent=True):
    sim.write_simulation(silent=silent)

# -------------------------------
# Runner (UNMASKED ONLY; stops after 30 SAVED successes)
# -------------------------------
def run_until_target_unmasked(prefix: str, target_success: int, max_attempts: int = 5000):
    successes = count_unmasked_successes(prefix)
    print(f"\n[{prefix}] UNMASKED successes (from REALZ_UNMASKED): {successes}/{target_success}")

    if successes >= target_success:
        print(f"[{prefix}] Already have enough UNMASKED successes.")
        print(get_unmasked_success_names(prefix))
        return

    i = next_new_index_from_unmasked(prefix)
    attempts = 0

    while successes < target_success and attempts < max_attempts:
        sim_name = f"{prefix}{i:03d}"
        sim_ws = workspace / sim_name

        # Do not overwrite an already-saved unmasked success
        if (REALZ_UNMASKED / sim_name / "K3d.npy").exists():
            i += 1
            continue

        # If a run folder exists but is partial, skip it (don’t reuse)
        if sim_ws.exists() and is_partial(sim_ws):
            print(f"⏩ Skipping partial/non-terminated run folder: {sim_name}")
            i += 1
            continue

        seed = _seed(i)
        K3d = make_K_realization_unmasked(seed=seed)

        inflow = float(parameters["Scenario_A"]["inflow"])
        sim = build_models(sim_name, K3d=K3d, cat3d=cat_base, inflow=inflow, geo=geo_A)

        if WRITE:
            write_models(sim, silent=True)

        if RUN:
            _ok, _buff = sim.run_simulation(silent=False, report=True)

        attempts += 1

        if outputs_true_success(sim_ws, T_FINAL):
            # ✅ Save ONLY after MF6 success
            save_unmasked_realization(sim_name, K3d, seed=seed)
            if PLOT:
                plot_K_heatmap(sim_name, K3d, top_window_m=300.0)

            successes += 1
            print(f"✅ TRUE Success SAVED: {sim_name} ({successes}/{target_success})")
        else:
            print(f"❌ Not a TRUE success (NOT saved): {sim_name} ({successes}/{target_success})")

        i += 1

    print(f"\n[{prefix}] DONE. UNMASKED successes = {successes}/{target_success}")
    print(get_unmasked_success_names(prefix))

# -------------------------------
# RUN (UNMASKED ONLY)
# -------------------------------
run_until_target_unmasked("Scenario_A_nomask_R", target_success=30)


[Scenario_A_nomask_R] UNMASKED successes (from REALZ_UNMASKED): 29/30
FloPy is using the following executable to run the model: ..\..\..\..\..\EXE\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.5.0 05/23/2024

   MODFLOW 6 compiled May 23 2024 18:06:57 with Intel(R) Fortran Intel(R) 64
   Compiler Classic for applications running on Intel(R) 64, Version 2021.7.0
                             Build 20220726_000000

This software has been approved for release by the U.S. Geological 
Survey (USGS). Although the software has been subjected to rigorous 
review, the USGS reserves the right to update the software as needed 
pursuant to further analysis and review. No warranty, expressed or 
implied, is made by the USGS or the U.S. Government as to the 
functionality of the software and related material nor shall the 
fact of release constitute any such warranty. Furthermore, the 
softwa